# 离职预测与影响因素分析

本笔记本围绕 HR_comma_sep.csv 完成数据加载、清洗、探索性分析，并构建基线与提升模型，给出关键特征的重要性与管理建议可视化。

## 1. 数据加载与目标

- 明确目标：离职预测、影响因素分析、输出管理建议线索
- 加载数据：读取 `HR_comma_sep.csv`，如不存在则合成与保存
- 检查行列数量与字段类型

In [ ]:

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

np.random.seed(42)
sns.set_theme(style="whitegrid")

def expit(x):
    return 1 / (1 + np.exp(-x))

data_path = Path("..") / "data" / "HR_comma_sep.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)


In [ ]:

if not data_path.exists():
    rng = np.random.default_rng(42)
    n = 15000
    sales_options = [
        "sales",
        "accounting",
        "hr",
        "technical",
        "support",
        "management",
        "IT",
        "product_mng",
        "marketing",
        "RandD",
    ]
    salaries = rng.choice(["low", "medium", "high"], size=n, p=[0.5, 0.35, 0.15])
    satisfaction_level = rng.beta(2, 5, size=n)
    last_evaluation = rng.beta(2, 2, size=n)
    number_project = rng.integers(2, 8, size=n)
    avg_hours = rng.normal(200, 30, size=n)
    avg_hours = np.clip(avg_hours, 90, 320)
    time_spend_company = rng.integers(1, 11, size=n)
    work_accident = rng.binomial(1, 0.1, size=n)
    promotion_last_5years = rng.binomial(1, 0.05 + 0.05 * (salaries == "high"))
    sales = rng.choice(sales_options, size=n)

    logit = (
        -3
        - 3 * (satisfaction_level - 0.5)
        + 0.8 * (avg_hours > 220)
        + 0.5 * (number_project >= 6)
        + 0.4 * (time_spend_company >= 6)
        + 0.6 * (salaries == "low")
        - 0.4 * (salaries == "high")
        + 0.5 * (last_evaluation < 0.5)
        - 0.5 * promotion_last_5years
    )
    attrition_prob = expit(logit)
    left = rng.binomial(1, attrition_prob)

    df = pd.DataFrame(
        {
            "satisfaction_level": satisfaction_level,
            "last_evaluation": last_evaluation,
            "number_project": number_project,
            "average_monthly_hours": avg_hours,
            "time_spend_company": time_spend_company,
            "Work_accident": work_accident,
            "left": left,
            "promotion_last_5years": promotion_last_5years,
            "sales": sales,
            "salary": salaries,
        }
    )
    df.to_csv(data_path, index=False)

hr_df = pd.read_csv(data_path)
print(f"数据路径: {data_path.resolve()}")
print("数据集形状 (行, 列):", hr_df.shape)
print("字段类型:
", hr_df.dtypes)
hr_df.head()


## 2. 数据清洗

- 检查缺失值、重复值
- 关注工时/项目数极端值并裁剪

In [ ]:

missing = hr_df.isna().sum()
print("缺失值统计:
", missing)
print("重复行数量:", hr_df.duplicated().sum())

caps = {
    "average_monthly_hours": hr_df["average_monthly_hours"].quantile([0.01, 0.99]).values,
    "number_project": hr_df["number_project"].quantile([0.01, 0.99]).values,
}

hr_df_clean = hr_df.copy()
for col, (low, high) in caps.items():
    before = hr_df_clean[col].describe()
    hr_df_clean[col] = hr_df_clean[col].clip(lower=low, upper=high)
    after = hr_df_clean[col].describe()
    print(f"
列 {col} 裁剪前后主要统计量变化:
前:
{before}
后:
{after}")

hr_df_clean.describe()


## 3. 初步 EDA

- 总体与按 `left` 分组的描述性统计
- 关键变量分布与分组箱线图

In [ ]:

overall_stats = hr_df_clean.describe()
by_left_stats = hr_df_clean.groupby("left").describe().T
print("总体描述性统计:
", overall_stats)
print("
按是否离职分组的描述性统计 (转置后便于阅读):
", by_left_stats)


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(hr_df_clean, x="satisfaction_level", hue="left", multiple="stack", bins=30, ax=axes[0, 0])
axes[0, 0].set_title("满意度分布")

sns.histplot(hr_df_clean, x="average_monthly_hours", hue="left", multiple="stack", bins=30, ax=axes[0, 1])
axes[0, 1].set_title("月均工时分布")

sns.boxplot(data=hr_df_clean, x="left", y="number_project", ax=axes[1, 0])
axes[1, 0].set_title("项目数量 vs 离职")
axes[1, 0].set_xlabel("left (1=离职)")

sns.boxplot(data=hr_df_clean, x="left", y="time_spend_company", ax=axes[1, 1])
axes[1, 1].set_title("工龄 vs 离职")
axes[1, 1].set_xlabel("left (1=离职)")

plt.tight_layout()
plt.show()


## 4. 关联分析

- 数值特征相关系数矩阵
- 部门与工资等级的离职率差异

In [ ]:

num_cols = hr_df_clean.select_dtypes(include="number").columns
corr_matrix = hr_df_clean[num_cols].corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, cmap="crest", center=0, linewidths=0.5)
plt.title("数值特征相关系数矩阵")
plt.show()

dept_attrition = hr_df_clean.groupby("sales")["left"].mean().sort_values()
salary_attrition = hr_df_clean.groupby("salary")["left"].mean().sort_index()

print("部门离职率:
", dept_attrition)
print("
工资等级离职率:
", salary_attrition)


## 5. 特征预处理与数据集划分

- 对 `sales`、`salary` 做独热编码
- 8:2 划分训练 / 测试集

In [ ]:

X = hr_df_clean.drop(columns=["left"])
y = hr_df_clean["left"]

categorical_features = ["sales", "salary"]
numerical_features = X.columns.difference(categorical_features)

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
category_transformer = Pipeline(steps=[("encoder", OneHotEncoder(drop="first"))])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", category_transformer, categorical_features),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("训练集大小:", X_train.shape, "测试集大小:", X_test.shape)


## 6. 基线模型：逻辑回归

- 训练逻辑回归
- 输出准确率、混淆矩阵、Precision/Recall/F1

In [ ]:

log_reg_clf = Pipeline(
    steps=[("preprocess", preprocess), ("model", LogisticRegression(max_iter=200))]
)
log_reg_clf.fit(X_train, y_train)

log_pred = log_reg_clf.predict(X_test)
log_proba = log_reg_clf.predict_proba(X_test)[:, 1]

log_acc = accuracy_score(y_test, log_pred)
log_prec = precision_score(y_test, log_pred)
log_rec = recall_score(y_test, log_pred)
log_f1 = f1_score(y_test, log_pred)
log_auc = roc_auc_score(y_test, log_proba)

print(
    f"逻辑回归准确率: {log_acc:.3f}, Precision: {log_prec:.3f}, Recall: {log_rec:.3f}, F1: {log_f1:.3f}, ROC-AUC: {log_auc:.3f}"
)
print("
分类报告:
", classification_report(y_test, log_pred))
print("混淆矩阵:
", confusion_matrix(y_test, log_pred))


## 7. 提升模型：随机森林 + 交叉验证

- 5 折交叉验证比较 AUC / F1
- 网格搜索调参 (树数量、深度、最小分裂样本)

In [ ]:

rf_clf = Pipeline(
    steps=[("preprocess", preprocess), ("model", RandomForestClassifier(random_state=42))]
)

cv_auc = cross_val_score(rf_clf, X, y, cv=5, scoring="roc_auc")
cv_f1 = cross_val_score(rf_clf, X, y, cv=5, scoring="f1")
print(
    f"随机森林 5 折 AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}; F1: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}"
)

param_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [None, 12, 18],
    "model__min_samples_split": [2, 5],
}

grid_search = GridSearchCV(
    rf_clf,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring="roc_auc",
)

grid_search.fit(X_train, y_train)
print("最佳参数:", grid_search.best_params_)

best_rf = grid_search.best_estimator_
rf_pred = best_rf.predict(X_test)
rf_proba = best_rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_prec = precision_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

print(
    f"随机森林测试集- 准确率: {rf_acc:.3f}, Precision: {rf_prec:.3f}, Recall: {rf_rec:.3f}, F1: {rf_f1:.3f}, ROC-AUC: {rf_auc:.3f}"
)
print("
分类报告:
", classification_report(y_test, rf_pred))
print("混淆矩阵:
", confusion_matrix(y_test, rf_pred))


## 8. 特征重要性与模型解释

- 提取树模型 `feature_importances_`
- 讨论满意度、工时、工龄、晋升、工资等影响

In [ ]:

# 获取特征名顺序
num_features = numerical_features.tolist()
cat_encoder = best_rf.named_steps["preprocess"].named_transformers_["cat"].named_steps[
    "encoder"
]
cat_features = cat_encoder.get_feature_names_out(categorical_features)
feature_names = num_features + list(cat_features)

rf_model = best_rf.named_steps["model"]
importance_df = pd.DataFrame(
    {"feature": feature_names, "importance": rf_model.feature_importances_}
).sort_values("importance", ascending=False)

print("特征重要性(前10):
", importance_df.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), x="importance", y="feature", palette="viridis")
plt.title("随机森林特征重要性")
plt.tight_layout()
plt.show()


## 9. ROC 曲线对比

- 绘制逻辑回归与随机森林 ROC 曲线并标注 AUC

In [ ]:

log_fpr, log_tpr, _ = roc_curve(y_test, log_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

plt.figure(figsize=(8, 6))
plt.plot(log_fpr, log_tpr, label=f"逻辑回归 (AUC={log_auc:.3f})")
plt.plot(rf_fpr, rf_tpr, label=f"随机森林 (AUC={rf_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC 曲线对比")
plt.legend()
plt.tight_layout()
plt.show()


## 10. KMeans 聚类：识别高风险群体

- 选取核心特征做标准化后聚类
- 观察各簇的特征轮廓与离职率，标注高风险群体


In [ ]:
core_cols = [
    "satisfaction_level",
    "last_evaluation",
    "average_monthly_hours",
    "number_project",
    "time_spend_company",
    "promotion_last_5years",
]
cluster_df = hr_df_clean[core_cols].copy()
cluster_df["salary_level"] = hr_df_clean["salary"].map({"low": 0, "medium": 1, "high": 2})

cluster_scaler = StandardScaler()
cluster_scaled = cluster_scaler.fit_transform(cluster_df)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(cluster_scaled)
hr_df_clean["cluster"] = clusters

silhouette = silhouette_score(cluster_scaled, clusters)
overall_attrition = hr_df_clean["left"].mean()

cluster_summary = (
    hr_df_clean.groupby("cluster")
    .agg(
        样本数=("left", "size"),
        离职率=("left", "mean"),
        满意度=("satisfaction_level", "mean"),
        工时=("average_monthly_hours", "mean"),
        项目数=("number_project", "mean"),
        工龄=("time_spend_company", "mean"),
        晋升率=("promotion_last_5years", "mean"),
        低薪占比=("salary", lambda s: (s == "low").mean()),
    )
    .sort_index()
)
cluster_summary["风险标记"] = np.where(
    cluster_summary["离职率"] > overall_attrition * 1.2,
    "高风险群体",
    "相对稳定",
)

print(f"整体离职率: {overall_attrition:.3f}; Silhouette 系数: {silhouette:.3f}")
cluster_summary


In [ ]:
plot_df = cluster_summary.reset_index().rename(columns={"cluster": "簇"})
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=plot_df, x="簇", y="离职率", hue="风险标记", palette="flare", ax=axes[0])
axes[0].set_title("各簇离职率")
axes[0].axhline(overall_attrition, color="gray", linestyle="--", label="整体")
axes[0].legend()

profile_cols = ["满意度", "工时", "项目数", "工龄", "晋升率", "低薪占比"]
sns.heatmap(
    plot_df.set_index("簇")[profile_cols],
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    ax=axes[1],
)
axes[1].set_title("簇特征轮廓")
plt.tight_layout()
plt.show()


## 11. 管理建议与改进方向

- 关注标记为“高风险群体”的簇：满意度低、工时长、项目多或晋升率低、低薪比例高。
- 建议措施：
  - **补偿与奖励**：为低薪或高工时团队提供加班调休、绩效奖金、透明的晋升路径。
  - **工作负载管理**：降低高项目/高工时簇的任务并改善排班，防止长期超负荷。
  - **成长与反馈**：针对低满意度人群建立定期 1:1 反馈、培训与职业发展计划。
- 归纳主要影响因素：满意度、月均工时、项目数、工龄及晋升记录、薪资水平。通过提高满意度、优化工时与项目分配、加速晋升通道，可降低高风险簇的离职率。
